# Data Science for Business - Polynomials, Step Functions and Splines on Ames Housing

*Session 6 · ISLP 7.1 Polynomial Regression · 7.2 Step Functions · 7.3 Basis Functions · 7.4 Regression Splines*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

import statsmodels.formula.api as smf

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_06/ameshousing.csv')

## Prepare data

`MSSubClass` (type of dwelling) is stored as a number, but its values are codes for categories (e.g., 20 = 1-story house built 1946 or later, 60 = 2-story house built 1946 or later; see the data description). We therefore add the prefix `MSSC` to all values, which turns the variable into a string, so that it is treated as categorical.

In [ ]:
data["MSSubClass"] = "MSSC" + data["MSSubClass"].astype(str)

Let us focus on the numerical variables only (this also excludes `MSSubClass`, which is now categorical).

In [ ]:
data = data.select_dtypes(include='number')

Finally, we split the data into a training set (`train`) and a test set (`test`). Since we fit our models with statsmodels formulas, we keep predictors and response together in one data frame.

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
train.head()

## Visualize the data

To be able to draw our models, we now focus on **one predictor**: the year in which the house was built (`YearBuilt`).

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

The relationship is clearly **not a straight line**: houses built before about 1950 sell at roughly the same prices, while newer houses become more and more expensive, especially those built after about 1990. In this notebook, we try several ways to capture such a curved relationship (ISLP 7.1-7.4). All choices (degree, cutpoints, knots) are made **by hand**, by looking at the data; how to choose them systematically with cross-validation follows in the smoothing splines and GAMs notebook.

To draw the fitted curves, we need a grid of years, from the oldest to the newest house in the training data:

In [ ]:
year_grid = pd.DataFrame({'YearBuilt': np.arange(train['YearBuilt'].min(), train['YearBuilt'].max() + 1)})
year_grid.head()

## Linear regression with one predictor

As a reference, we fit a straight line.

In [ ]:
mod_linear = smf.ols(formula='SalePrice ~ YearBuilt', data=train).fit()
print(mod_linear.summary2())

We draw the fitted line together with a **95% confidence band** (dashed), like in the figures of ISLP Chapter 7. Both functions come from statsmodels: `get_prediction(...)` computes the predictions of a fitted model for new data (here: the year grid) together with their uncertainty, and `.summary_frame(alpha=0.05)` turns them into a data frame. It contains the predicted mean (`mean`) and the lower and upper end (`mean_ci_lower`, `mean_ci_upper`) of the 95% confidence interval for every year on the grid. The band shows how uncertain the estimated curve is (roughly ±2 standard errors).

In [ ]:
predictions = mod_linear.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
plt.title('Linear')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

The straight line overestimates the prices of old houses and underestimates the prices of new houses.

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_linear.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_linear.predict(test)))

## Polynomial regression

Polynomial regression (ISLP 7.1) adds powers of the predictor as additional predictors, e.g., for degree 4:

$$\text{SalePrice} = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + \beta_4 x^4 + \varepsilon$$

The model is still fitted by least squares; only the predictors are new. As in the book, we use degree 4.

One practical detail: $2010^4$ is about $1.6 \cdot 10^{13}$. Such huge numbers cause numerical problems (remember the rupees in Session 4). We therefore count the years since 1870, just before the oldest house in the data (`YearBuilt - 1870`), so that the numbers stay small. This does not change the shape of the fitted curve. In the formula, `I(...)` tells statsmodels to compute the expression in brackets before using it as a predictor.

In [ ]:
mod_poly = smf.ols(
    formula='SalePrice ~ I(YearBuilt - 1870) + I((YearBuilt - 1870)**2) + I((YearBuilt - 1870)**3) + I((YearBuilt - 1870)**4)',
    data=train
).fit()
mod_poly.params

The individual coefficients are hard to interpret; what matters is the fitted curve (ISLP 7.1).

In [ ]:
predictions = mod_poly.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
plt.title('Degree-4 polynomial')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

The curve follows the flat part for old houses and the steep rise for new houses. Note that the confidence band becomes wider at the left end, where there are only few (very old) houses. This is typical for polynomials: at the boundaries of the data, the curve can become very uncertain, and with higher degrees it can bend in strange ways. In practice, degrees above 3 or 4 are rarely used (ISLP 7.1).

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_poly.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_poly.predict(test)))

## Step functions

A step function (ISLP 7.2) cuts the range of the predictor into intervals and fits a **constant** (the average price) in each interval. We choose the cutpoints by hand: 1920, 1950, 1980 and 2000. `pd.cut` assigns every house to its interval:

In [ ]:
year_bins = [1870, 1920, 1950, 1980, 2000, 2010]

pd.cut(train['YearBuilt'], bins=year_bins).value_counts().sort_index()

`(1950, 1980]` means "built after 1950 and up to 1980". The intervals are a categorical predictor, so statsmodels turns them into dummy variables (`C(...)`), with the first interval as the baseline (ISLP 3.3.1). Since the cutpoints are fixed, the test data will be cut in exactly the same way.

In [ ]:
mod_step = smf.ols(formula='SalePrice ~ C(pd.cut(YearBuilt, bins=year_bins))', data=train).fit()
mod_step.params

The intercept is the average price of houses built up to 1920; each other coefficient is the price difference of its interval compared with this baseline.

In [ ]:
predictions = mod_step.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
plt.title('Step function')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

Step functions are easy to explain ("houses built between 1980 and 2000 cost on average about 89,000 dollars more than houses built up to 1920"), but they miss trends **within** an interval: all houses built after 2000 get the same predicted price, although prices clearly rise within this period. And the result depends on where we put the cutpoints.

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_step.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_step.predict(test)))

## Regression splines

Regression splines (ISLP 7.4) combine both ideas: we cut the range of the predictor at **knots**, fit a separate (low-degree) polynomial in each piece, and make sure the pieces fit together at the knots. We go from simple to more complex splines, always with a knot at 1980, where the scatter plot starts to rise.

The trick is a **hinge function** (in the book: *truncated power basis function*, ISLP 7.4.3). For a knot at 1980, `np.maximum(YearBuilt - 1980, 0)` is 0 for all houses built up to 1980 and counts the years since 1980 for newer houses:

In [ ]:
example = pd.DataFrame({'YearBuilt': [1960, 1970, 1980, 1990, 2000, 2010]})
example['hinge_1980'] = np.maximum(example['YearBuilt'] - 1980, 0)
example

### Linear spline

If we add the hinge function as a second predictor, the line can **change its slope** at the knot, while the two pieces stay connected.

In [ ]:
mod_linear_spline = smf.ols(formula='SalePrice ~ YearBuilt + I(np.maximum(YearBuilt - 1980, 0))', data=train).fit()

The coefficients have a simple interpretation: up to 1980, every year newer adds the coefficient of `YearBuilt` to the price. After 1980, the slope is the **sum** of both coefficients (about 500 + 3,100 ≈ 3,600 dollars per year, see column `Coef.`).

In [ ]:
print(mod_linear_spline.summary2())

The dashed vertical line marks the knot.

In [ ]:
predictions = mod_linear_spline.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
plt.axvline(1980, color='black', linestyle='--', linewidth=1)
plt.title('Linear spline (knot at 1980)')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_linear_spline.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_linear_spline.predict(test)))

### Continuous piecewise cubic

Instead of straight lines, we now want a **cubic** polynomial in each piece. We start with a cubic polynomial ($x$, $x^2$, $x^3$, again counting the years since 1870) and add the hinge function $h$, its square $h^2$ and its cube $h^3$. Before the knot, all hinge terms are 0, so only the cubic polynomial counts. After the knot, the three hinge terms change the shape of the curve. Since all hinge terms are 0 at the knot, the two pieces are connected (*continuous*): the curve cannot jump.

In [ ]:
mod_cubic_continuous = smf.ols(
    formula='SalePrice ~ I(YearBuilt - 1870) + I((YearBuilt - 1870)**2) + I((YearBuilt - 1870)**3)'
            '          + I(np.maximum(YearBuilt - 1980, 0)) + I(np.maximum(YearBuilt - 1980, 0)**2) + I(np.maximum(YearBuilt - 1980, 0)**3)',
    data=train
).fit()

In [ ]:
predictions = mod_cubic_continuous.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
plt.axvline(1980, color='black', linestyle='--', linewidth=1)
plt.title('Continuous piecewise cubic (knot at 1980)')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

The curve does not jump at 1980, but it has a visible **bend** there: the price trend changes abruptly from one year to the next. That is not very plausible for houses built in 1980 and 1981.

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_cubic_continuous.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_cubic_continuous.predict(test)))

### Cubic spline

To get rid of the bend, a **cubic spline** keeps the cubic polynomial but uses **only the cubed hinge function** $h^3$. Why does this make the curve smooth at the knot? Let's look at how the three hinge functions start at the knot:

In [ ]:
hinge_grid = np.arange(1960, 2001)
hinge = np.maximum(hinge_grid - 1980, 0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(hinge_grid, hinge, linewidth=2)
axes[0].set_title('h')

axes[1].plot(hinge_grid, hinge**2, linewidth=2)
axes[1].set_title('h²')

axes[2].plot(hinge_grid, hinge**3, linewidth=2)
axes[2].set_title('h³')

for ax in axes:
    ax.axvline(1980, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('YearBuilt')

plt.tight_layout()
plt.show()

All three functions are 0 up to the knot, but they "take off" in different ways:

- $h$ takes off with a **sharp corner**: right after the knot it rises with a fixed slope. Adding $h$ to a curve therefore creates a bend at the knot. This is what we saw in the continuous piecewise cubic.
- $h^2$ takes off **flat** (no corner), but it starts bending upwards right away: the curvature of the curve changes abruptly at the knot.
- $h^3$ takes off **flat and without curvature**: it leaves 0 so gently that, at the knot, the curve keeps its value, its slope and its curvature. Only after the knot does the curve gradually bend differently.

So with only $h^3$, the two pieces are not only connected, they join **smoothly** (mathematically: the first and second derivatives are continuous, ISLP 7.4.2). You cannot see where one piece ends and the next begins. As a side effect, the cubic spline needs two coefficients fewer than the continuous piecewise cubic.

In [ ]:
mod_cubic_spline = smf.ols(
    formula='SalePrice ~ I(YearBuilt - 1870) + I((YearBuilt - 1870)**2) + I((YearBuilt - 1870)**3) '
    '                  + I(np.maximum(YearBuilt - 1980, 0)**3)',
    data=train
).fit()

In [ ]:
predictions = mod_cubic_spline.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
plt.axvline(1980, color='black', linestyle='--', linewidth=1)
plt.title('Cubic spline (knot at 1980)')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_cubic_spline.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_cubic_spline.predict(test)))

Interestingly, the test RMSE of the cubic spline is somewhat higher than that of the continuous piecewise cubic: for our data, the abrupt change at 1980 happens to fit a bit better than a smooth transition with only one knot. Smoothness makes a curve more plausible, but it does not guarantee a lower error. With more knots, the cubic spline gets more flexibility, as we will see next.

### Cubic spline with several knots

With more knots, the spline becomes more flexible. A cubic spline with $K$ knots consists of $x$, $x^2$, $x^3$ and one cubed hinge function per knot (ISLP 7.4.3). We place three knots by hand, at 1950, 1980 and 2000, where the shape of the scatter plot seems to change.

In [ ]:
mod_spline_3knots = smf.ols(
    formula='SalePrice ~ I(YearBuilt - 1870) + I((YearBuilt - 1870)**2) + I((YearBuilt - 1870)**3)'
            '          + I(np.maximum(YearBuilt - 1950, 0)**3) '
            '          + I(np.maximum(YearBuilt - 1980, 0)**3) '
            '          + I(np.maximum(YearBuilt - 2000, 0)**3)',
    data=train
).fit()

In [ ]:
predictions = mod_spline_3knots.get_prediction(year_grid).summary_frame(alpha=0.05)

plt.figure(figsize=(8, 5))
plt.scatter(train['YearBuilt'], train['SalePrice'], s=10, color='gray', alpha=0.3)
plt.plot(year_grid['YearBuilt'], predictions['mean'], color='blue', linewidth=2)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_lower'], color='blue', linestyle='--', linewidth=1)
plt.plot(year_grid['YearBuilt'], predictions['mean_ci_upper'], color='blue', linestyle='--', linewidth=1)
for knot in [1950, 1980, 2000]:
    plt.axvline(knot, color='black', linestyle='--', linewidth=1)
plt.title('Cubic spline (knots at 1950, 1980, 2000)')
plt.xlabel('YearBuilt')
plt.ylabel('SalePrice')
plt.show()

In [ ]:
print('Train RMSE:', root_mean_squared_error(train['SalePrice'], mod_spline_3knots.predict(train)))
print('Test RMSE: ', root_mean_squared_error(test['SalePrice'], mod_spline_3knots.predict(test)))

We chose the knots by looking at the data. Alternatively, knots can be placed at quantiles of the predictor, and their number can be chosen by cross-validation (ISLP 7.4.4). The book also introduces *natural* splines, which are additionally forced to be linear beyond the outer knots and are therefore more stable at the boundaries.

## Compare the models

We collect the test RMSEs of all models.

In [ ]:
comparison = pd.DataFrame({
    'Model': [
        'Linear',
        'Degree-4 polynomial',
        'Step function',
        'Linear spline (1 knot)',
        'Continuous piecewise cubic (1 knot)',
        'Cubic spline (1 knot)',
        'Cubic spline (3 knots)',
    ],
    'Test RMSE': [
        root_mean_squared_error(test['SalePrice'], mod_linear.predict(test)),
        root_mean_squared_error(test['SalePrice'], mod_poly.predict(test)),
        root_mean_squared_error(test['SalePrice'], mod_step.predict(test)),
        root_mean_squared_error(test['SalePrice'], mod_linear_spline.predict(test)),
        root_mean_squared_error(test['SalePrice'], mod_cubic_continuous.predict(test)),
        root_mean_squared_error(test['SalePrice'], mod_cubic_spline.predict(test)),
        root_mean_squared_error(test['SalePrice'], mod_spline_3knots.predict(test)),
    ]
})
comparison.round(0)

All non-linear versions predict the test data better than the straight line. The cubic spline with three knots has the lowest test RMSE.

Keep in mind that the differences are moderate: the year of construction alone explains only a part of the price differences between houses.

## Your turn!

**Your task:** Model the relationship between `SalePrice` and the above-ground living area (`GrLivArea`) with one predictor. Plot the data first, then try a polynomial, a step function and a spline with knots of your choice. Which model has the lowest test RMSE?

In [ ]:
# YOUR CODE HERE